In [ ]:
%sql
-- 1. Crear el catálogo del equipo
CREATE CATALOG IF NOT EXISTS stasys_team2;

-- 2. Seleccionar el catálogo
USE CATALOG stasys_team2;

-- ----------------------------------------------------
-- ESQUEMA DE TEAM 02: 
-- ----------------------------------------------------

-- 3. Crear tu esquema de trabajo
CREATE SCHEMA IF NOT EXISTS gianmarco;

-- 4. Verificar los esquemas disponibles
SHOW SCHEMAS IN stasys_team2;

-- 5. Comprobar permisos y ubicación
SELECT current_catalog(), current_schema();

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS
    stasys_team2.gianmarco.consultas_denuncias_silver
USING CSV
OPTIONS (
    header = "true",
    inferSchema = "true"
)
LOCATION 's3://data-lagoon-y/silver/cleaned/consultas_denuncias_2026_v1/2026/10/13/20261013_150857_891495/CONSULTAS_DENUNCIAS_2026_v1_silver.csv';

In [ ]:
%sql
-- Confirmar que la tabla existe.
DESCRIBE TABLE
    stasys_team2.gianmarco.consultas_denuncias_silver;

-- Consultar una muestra.
SELECT *
FROM stasys_team2.gianmarco.consultas_denuncias_silver
LIMIT 20;

-- Contar los registros.
SELECT COUNT(*) AS total_registros
FROM stasys_team2.gianmarco.consultas_denuncias_silver;


In [ ]:
%sql
-- Consulta 1 — Cantidad de solicitudes por tipo

SELECT
    DE_TIPO_SOLICITUD AS tipo_solicitud,
    COUNT(*) AS total_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
GROUP BY DE_TIPO_SOLICITUD
ORDER BY total_solicitudes DESC;

In [ ]:
%sql
-- Consulta 2 — Evolución mensual de las solicitudes

SELECT
    DATE_FORMAT(TO_DATE(FE_REGISTRO), 'yyyy-MM') AS periodo,
    COUNT(*) AS total_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
WHERE TO_DATE(FE_REGISTRO) IS NOT NULL
GROUP BY DATE_FORMAT(TO_DATE(FE_REGISTRO), 'yyyy-MM')
ORDER BY periodo;

In [ ]:
%sql
-- Consulta 3 — Ranking de instituciones

SELECT
    CO_INSTITUCION,
    DE_INSTITUCION,
    COUNT(*) AS total_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
GROUP BY CO_INSTITUCION, DE_INSTITUCION
ORDER BY total_solicitudes DESC
LIMIT 10;


In [ ]:
%sql

-- Consulta 4 — Distribución por naturaleza de institución


SELECT
    DE_NATURALEZA_INST AS naturaleza_institucion,
    COUNT(*) AS total_solicitudes,
    COUNT(DISTINCT CO_INSTITUCION) AS instituciones_distintas
FROM stasys_team2.gianmarco.consultas_denuncias_silver
GROUP BY DE_NATURALEZA_INST
ORDER BY total_solicitudes DESC;


In [ ]:
%sql
-- Consulta 5 — Distribución por edad y género


SELECT
    CASE
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) < 18
            THEN 'Menor de 18'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) BETWEEN 18 AND 29
            THEN '18-29'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) BETWEEN 30 AND 44
            THEN '30-44'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) BETWEEN 45 AND 59
            THEN '45-59'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) >= 60
            THEN '60 o más'
        ELSE 'Edad no disponible'
    END AS grupo_edad,
    DE_SOLICITANTE_GENERO AS genero,
    COUNT(*) AS total_solicitudes
FROM stasys_team2.gianmarco.consultas_denuncias_silver
GROUP BY
    CASE
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) < 18
            THEN 'Menor de 18'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) BETWEEN 18 AND 29
            THEN '18-29'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) BETWEEN 30 AND 44
            THEN '30-44'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) BETWEEN 45 AND 59
            THEN '45-59'
        WHEN TRY_CAST(DE_SOLICITANTE_EDAD AS INT) >= 60
            THEN '60 o más'
        ELSE 'Edad no disponible'
    END,
    DE_SOLICITANTE_GENERO
ORDER BY grupo_edad, genero;